# Лабораторная работа №2. Свёрточные нейронные сети (CNN)
**Курс:** Нейронные сети и глубокое обучение
**Студент:** Никита, группа 6412-100503D
**Данные:** CIFAR-10 (цветные изображения 32×32, 10 классов)

**Цель:** изучить принципы построения, обучения, оптимизации и оценки качества свёрточных нейронных сетей, освоить Transfer Learning.

## 0. Импорт библиотек

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.keras.utils.set_random_seed(42)
print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

## 1. Загрузка данных CIFAR-10

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

y_train = y_train.flatten()
y_test = y_test.flatten()

CLASS_NAMES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']
NUM_CLASSES = len(CLASS_NAMES)

print('x_train:', x_train.shape, x_train.dtype)
print('y_train:', y_train.shape)
print('x_test: ', x_test.shape)
print('y_test: ', y_test.shape)
print('Картинок каждого класса в train:', np.bincount(y_train))

### По 3 изображения каждого класса

In [ ]:
N_PER_CLASS = 3
fig, axes = plt.subplots(NUM_CLASSES, N_PER_CLASS, figsize=(4.5, 15))
for c in range(NUM_CLASSES):
    idx = np.where(y_train == c)[0][:N_PER_CLASS]
    for j, i in enumerate(idx):
        axes[c, j].imshow(x_train[i])
        axes[c, j].axis('off')
    axes[c, 0].set_title(CLASS_NAMES[c], fontsize=10, loc='left')
plt.tight_layout()
plt.show()

### Предобработка
Пиксели лежат в диапазоне 0..255 (uint8). Делим на 255 → диапазон 0..1 (float32): так сеть обучается быстрее и стабильнее.
Метки оставляем целыми числами 0..9 и используем функцию потерь `sparse_categorical_crossentropy` (one-hot кодирование не нужно).

In [ ]:
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0
print(x_train.min(), x_train.max())

## 2. Построение модели
Архитектура по заданию:
`Input(32×32×3) → [Conv2D → MaxPooling]×3 → Flatten → Dense(relu) → Dropout → Dense(softmax)`

Число фильтров: 32 → 64 → 128 (растёт с глубиной, т.к. растёт поле восприятия и «осмысленных» признаков становится больше).

In [ ]:
def build_model(num_classes, name='cnn'):
    model = keras.Sequential([
        keras.Input(shape=(32, 32, 3)),
        layers.Conv2D(32, (3, 3), padding='same', activation='relu', name='conv1'),
        layers.MaxPooling2D((2, 2), name='pool1'),
        layers.Conv2D(64, (3, 3), padding='same', activation='relu', name='conv2'),
        layers.MaxPooling2D((2, 2), name='pool2'),
        layers.Conv2D(128, (3, 3), padding='same', activation='relu', name='conv3'),
        layers.MaxPooling2D((2, 2), name='pool3'),
        layers.Flatten(name='flatten'),
        layers.Dense(128, activation='relu', name='fc'),
        layers.Dropout(0.5, name='dropout'),
        layers.Dense(num_classes, activation='softmax', name='output'),
    ], name=name)

    model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

model_10 = build_model(NUM_CLASSES, name='cnn_10class')
model_10.summary()

### Проверка числа обучаемых параметров
Для свёрточного слоя: **params = (M₁·M₂·K_in + 1) · K_out**
(M₁×M₂ — размер ядра, K_in — каналов на входе, +1 — bias, K_out — число фильтров).
Для полносвязного: **params = (N_in + 1) · N_out**. У MaxPooling, Flatten, Dropout параметров нет.

In [ ]:
def conv_params(m1, m2, k_in, k_out):
    return (m1 * m2 * k_in + 1) * k_out

def dense_params(n_in, n_out):
    return (n_in + 1) * n_out

p = [conv_params(3, 3, 3, 32),
     conv_params(3, 3, 32, 64),
     conv_params(3, 3, 64, 128),
     dense_params(4 * 4 * 128, 128),
     dense_params(128, 10)]
print('По слоям:', p)
print('Всего по формуле:', sum(p))
print('Всего по Keras:  ', model_10.count_params())

## Вспомогательные функции

In [ ]:
def make_callbacks(name):
    return [
        keras.callbacks.ModelCheckpoint(
            filepath=f'{name}_best.weights.h5',
            monitor='val_accuracy', mode='max',
            save_best_only=True, save_weights_only=True, verbose=1),
        keras.callbacks.CSVLogger(f'{name}_log.csv'),
    ]


def plot_history(history, title):
    h = history.history
    epochs = range(1, len(h['loss']) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(epochs, h['accuracy'], label='train')
    ax1.plot(epochs, h['val_accuracy'], label='validation')
    ax1.set_title(f'{title}: точность'); ax1.set_xlabel('эпоха'); ax1.legend(); ax1.grid(True)
    ax2.plot(epochs, h['loss'], label='train')
    ax2.plot(epochs, h['val_loss'], label='validation')
    ax2.set_title(f'{title}: потери'); ax2.set_xlabel('эпоха'); ax2.legend(); ax2.grid(True)
    plt.show()


def per_class_accuracy(y_true, y_pred, class_ids):
    return {c: 100 * np.mean(y_pred[y_true == c] == c) for c in class_ids}


def show_errors(x, y_true, y_pred, class_ids, n=4):
    fig, axes = plt.subplots(len(class_ids), n, figsize=(n * 1.8, len(class_ids) * 1.9))
    for row, c in enumerate(class_ids):
        wrong = np.where((y_true == c) & (y_pred != c))[0][:n]
        for j in range(n):
            ax = axes[row, j]
            ax.axis('off')
            if j < len(wrong):
                i = wrong[j]
                ax.imshow(x[i])
                ax.set_title(f'{CLASS_NAMES[c]}→{CLASS_NAMES[y_pred[i]]}', fontsize=8)
    plt.tight_layout()
    plt.show()

## 3. Обучение модели на 10 классах
`validation_split=0.2` — традиционные 20% обучающей выборки уходят на валидацию (40 000 train / 10 000 val).

In [ ]:
EPOCHS = 20
BATCH_SIZE = 64

history_10 = model_10.fit(
    x_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.2,
    callbacks=make_callbacks('cnn_10class'),
    verbose=2)

plot_history(history_10, '10 классов')

### Сохранение модели и загрузка с лучшими весами

In [ ]:
model_10.save('cnn_10class.keras')

model_10 = keras.models.load_model('cnn_10class.keras')
model_10.load_weights('cnn_10class_best.weights.h5')

test_loss, test_acc_10 = model_10.evaluate(x_test, y_test, verbose=0)
print(f'Точность на тесте (10 классов): {test_acc_10:.4f}')

### Классификация тестовой выборки, ошибки и точность по классам

In [ ]:
probs_10 = model_10.predict(x_test, verbose=0)
pred_10 = np.argmax(probs_10, axis=1)

acc10 = per_class_accuracy(y_test, pred_10, range(NUM_CLASSES))
for c, a in acc10.items():
    print(f'{c+1:2d} {CLASS_NAMES[c]:<11s} {a:5.1f}%')

show_errors(x_test, y_test, pred_10, list(range(NUM_CLASSES)))

## 4. Transfer Learning
### 4.1 Убираем 2 класса и обучаем модель на 8 классах

In [ ]:
REMOVED = [3, 5]
KEEP = [c for c in range(NUM_CLASSES) if c not in REMOVED]
print('Удалены:', [CLASS_NAMES[c] for c in REMOVED])
print('Оставлены:', [CLASS_NAMES[c] for c in KEEP])

old2new = np.full(NUM_CLASSES, -1)
old2new[KEEP] = np.arange(len(KEEP))
new2old = np.array(KEEP)

train_mask = np.isin(y_train, KEEP)
test_mask = np.isin(y_test, KEEP)

x_train8, y_train8 = x_train[train_mask], old2new[y_train[train_mask]]
x_test8, y_test8 = x_test[test_mask], old2new[y_test[test_mask]]

print('train:', x_train8.shape, ' test:', x_test8.shape)
print('новые метки:', np.unique(y_train8))

In [ ]:
model_8 = build_model(len(KEEP), name='cnn_8class')
model_8.summary()

history_8 = model_8.fit(
    x_train8, y_train8,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.2,
    callbacks=make_callbacks('cnn_8class'),
    verbose=2)

plot_history(history_8, '8 классов')

In [ ]:
model_8.save('cnn_8class.keras')
model_8 = keras.models.load_model('cnn_8class.keras')
model_8.load_weights('cnn_8class_best.weights.h5')

_, test_acc_8 = model_8.evaluate(x_test8, y_test8, verbose=0)
print(f'Точность на тесте (8 классов): {test_acc_8:.4f}')

pred_8 = new2old[np.argmax(model_8.predict(x_test8, verbose=0), axis=1)]
acc8 = per_class_accuracy(y_test[test_mask], pred_8, KEEP)
for c, a in acc8.items():
    print(f'{c+1:2d} {CLASS_NAMES[c]:<11s} {a:5.1f}%')

### 4.2 Удаляем полносвязные слои, замораживаем свёрточную часть
Последние слои модели: `fc (Dense) → dropout → output (Dense)`. Чтобы убрать оба полносвязных слоя, нужно 3 раза вызвать `pop()` (Dropout стоит между ними). Остаётся свёрточная часть до `Flatten` включительно — «экстрактор признаков».

In [ ]:
base = keras.models.load_model('cnn_8class.keras', compile=False)
base.load_weights('cnn_8class_best.weights.h5')

base.pop()
base.pop()
base.pop()
base.trainable = False

base.summary()

In [ ]:
model_tl = keras.Sequential([
    keras.Input(shape=(32, 32, 3)),
    base,
    layers.Dense(128, activation='relu', name='fc_new'),
    layers.Dropout(0.5, name='dropout_new'),
    layers.Dense(NUM_CLASSES, activation='softmax', name='output_new'),
], name='cnn_transfer_10class')

model_tl.compile(optimizer='adam',
                 loss='sparse_categorical_crossentropy',
                 metrics=['accuracy'])
model_tl.summary()

In [ ]:
history_tl = model_tl.fit(
    x_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.2,
    callbacks=make_callbacks('cnn_transfer_10class'),
    verbose=2)

plot_history(history_tl, 'Transfer Learning, 10 классов')

In [ ]:
model_tl.save('cnn_transfer_10class.keras')
model_tl = keras.models.load_model('cnn_transfer_10class.keras')
model_tl.load_weights('cnn_transfer_10class_best.weights.h5')

_, test_acc_tl = model_tl.evaluate(x_test, y_test, verbose=0)
print(f'Точность на тесте (Transfer Learning): {test_acc_tl:.4f}')

pred_tl = np.argmax(model_tl.predict(x_test, verbose=0), axis=1)
acc_tl = per_class_accuracy(y_test, pred_tl, range(NUM_CLASSES))
show_errors(x_test, y_test, pred_tl, REMOVED)

## 5. Итоговая таблица: % правильной классификации на тестовом наборе

In [ ]:
rows = []
for c in range(NUM_CLASSES):
    rows.append({
        'класс': f'{c+1} ({CLASS_NAMES[c]})',
        'Ini_10class': acc10[c],
        'Ini_8class': acc8.get(c, np.nan),
        'TransferLearning_10class': acc_tl[c],
    })
table = pd.DataFrame(rows).set_index('класс')
table.loc['Общая точность'] = [100 * test_acc_10, 100 * test_acc_8, 100 * test_acc_tl]
table.round(1).fillna('—')

## 6. Выводы
*(подставь цифры из своей таблицы)*

1. Свёрточная сеть из трёх блоков Conv+MaxPool при ~357 тыс. параметров даёт на CIFAR-10 точность порядка 70–75%. Хуже всего распознаются cat, dog, bird — они похожи между собой, лучше всего — automobile, ship, truck, frog.
2. По графикам видно, что после ~8–12 эпох validation loss перестаёт падать и начинает расти, а train-точность продолжает расти — начинается переобучение. Поэтому важно сохранять лучшие веса через `ModelCheckpoint`, а не брать веса последней эпохи.
3. Модель на 8 классах показывает более высокую общую точность: задача проще, и убраны самые «путающиеся» классы (cat/dog).
4. Transfer Learning: свёрточные слои обучались без cat и dog, но их признаки (края, текстуры, формы) универсальны, поэтому дообученная модель распознаёт все 10 классов. Обучались только новые полносвязные слои (~264 тыс. параметров вместо ~357 тыс.), эпоха проходит быстрее.
5. Точность на «новых» классах (cat, dog) у Transfer Learning ниже, чем у модели, обученной на 10 классах с нуля: замороженные свёртки не учились выделять признаки, отличающие кошку от собаки. Общая точность TL обычно немного ниже Ini_10class. Улучшить можно fine-tuning'ом — разморозить последний свёрточный слой и дообучить с маленьким learning rate.